In [20]:
from wien2k import *

"""
MnCrAs doubled
1.0 Ang
3.5779930000000002 0.0000000000000002 0.0000000000000002
0.0000000000000000 3.5779930000000002 0.0000000000000002
0.0000000000000000 0.0000000000000000 12.2525300000000001
Cr Mn As 
4 4 4 
Direct
0.7500000000000000 0.7500000000000000 0.3326450000000000 Cr
0.2500000000000000 0.2500000000000000 0.1673550000000000 Cr
0.7500000000000000 0.7500000000000000 0.8326450000000001 Cr
0.2500000000000000 0.2500000000000000 0.6673550000000000 Cr
0.7500000000000000 0.2500000000000000 0.0000000000000000 Mn
0.2500000000000000 0.7500000000000000 0.0000000000000000 Mn
0.7500000000000000 0.2500000000000000 0.5000000000000000 Mn
0.2500000000000000 0.7500000000000000 0.5000000000000000 Mn
0.7500000000000000 0.7500000000000000 0.1359680000000000 As
0.2500000000000000 0.2500000000000000 0.3640320000000000 As
0.7500000000000000 0.7500000000000000 0.6359680000000000 As
0.2500000000000000 0.2500000000000000 0.8640320000000000 As
"""

mncras = """
Mn2 Cr2 As2
1.0 Ang
   3.5779930000000002    0.0000000000000000    0.0000000000000002
   0.0000000000000006    3.5779930000000002    0.0000000000000002
   0.0000000000000000    0.0000000000000000    6.1262650000000001
Mn Cr As
2 2 2
direct
   0.0000000000000000    0.0000000000000000    0.0000000000000000 Mn
   0.5000000000000000    0.5000000000000000    0.0000000000000000 Mn
   0.0000000000000000    0.5000000000000000    0.6652900000000000 Cr
   0.5000000000000000    0.0000000000000000    0.3347100000000000 Cr
   0.0000000000000000    0.5000000000000000    0.2719360000000001 As
   0.5000000000000000    0.0000000000000000    0.7280639999999998 As

"""

# load the structure from materials project
# struct = StructureFile.load_materials_project(
#     "https://next-gen.materialsproject.org/materials/mp-1221644?formula=CrMnAs",  # load in MnCrAs
#     "credentials.json",
# )
struct = StructureFile.parse_poscar(mncras)
struct.tweak_cell_multiples(c=2)
orig_poscar = struct.generate_poscar()

print("Original POSCAR loaded")
print(orig_poscar)

Original POSCAR loaded
Mn2 Cr2 As2
1.0 Ang
3.5779930000000002 0.0000000000000002 0.0000000000000002
0.0000000000000000 3.5779930000000002 0.0000000000000002
0.0000000000000000 0.0000000000000000 12.2525300000000001
Cr Mn As 
4 4 4 
Direct
0.0000000000000000 0.5000000000000000 0.3326450000000000 Cr
0.5000000000000000 0.0000000000000000 0.1673550000000000 Cr
0.0000000000000000 0.5000000000000000 0.8326450000000001 Cr
0.5000000000000000 0.0000000000000000 0.6673550000000000 Cr
0.0000000000000000 0.0000000000000000 0.0000000000000000 Mn
0.5000000000000000 0.5000000000000000 0.0000000000000000 Mn
0.0000000000000000 0.0000000000000000 0.5000000000000000 Mn
0.5000000000000000 0.5000000000000000 0.5000000000000000 Mn
0.0000000000000000 0.5000000000000000 0.1359680000000001 As
0.5000000000000000 0.0000000000000000 0.3640319999999999 As
0.0000000000000000 0.5000000000000000 0.6359680000000001 As
0.5000000000000000 0.0000000000000000 0.8640319999999999 As



In [21]:
# find valid permutations (8 choose 4) for CrMnAs case
valid_combinations = []
for i in range(2**8):
    bin_rep = f"{i:08b}"

    if bin_rep.count("0") == 4:
        # passes combination check
        valid_combinations.append(bin_rep)

print("Valid combinations found", len(valid_combinations))

Valid combinations found 70


In [22]:
# generate all the valid combination structures
combination_structures = []
for comb in valid_combinations:
    struct_copy = StructureFile.parse_poscar(orig_poscar)

    for i in range(8):
        struct_copy.tweak_atom(i, Z=(24 if comb[i] == "0" else 25))  # Cr or Mn

    combination_structures.append(struct_copy)

print("Combination structures generated", len(combination_structures))

Combination structures generated 70


In [23]:
equivalence_matrix = np.zeros(
    (len(combination_structures), len(combination_structures))
).astype(int)
for i in range(len(combination_structures)):
    s1 = combination_structures[i]

    for j in range(len(combination_structures)):
        s2 = combination_structures[j]

        # print(i, j)
        # if s1 != s2:
        #     are_equiv, proof = s1.translational_equivalence_check(s2)
        #     print(are_equiv, proof)

        #     if are_equiv:
        #         equivalence_matrix[i][j] = 1
        # else:
        #     equivalence_matrix[i][j] = 1
        
        are_equiv, proof = s1.translational_equivalence_check(s2)
        
        if are_equiv:
            equivalence_matrix[i][j] = 1

with open("_equivalence_matrix.txt", "w+") as writer:
    writer.write(        
        "\n".join(
            ["".join(["{:2}".format(item) for item in row]) for row in equivalence_matrix]
        )
    )
    
print("Equivalent structures found")

Equivalent structures found


In [24]:
rows_to_check = list(range(len(combination_structures)))

unique_structures = []
necessary_structures = []
while len(rows_to_check) > 0:
    r = rows_to_check[0]
    
    necessary_structures.append(combination_structures[r])
    print(valid_combinations[r])
    
    count = 0
    for c in range(len(equivalence_matrix[r])):
        if equivalence_matrix[r][c] == 1:
            rows_to_check.remove(c)
            count+=1 
            
    if count == 1:
        unique_structures.append(combination_structures[r])
    
print("Unique structures found", len(unique_structures), "from", len(combination_structures))
print("Necessary structures found", len(necessary_structures), "from", len(combination_structures))

00001111
00010111
00011011
00011101
00011110
00100111
00101011
00101101
00101110
00110011
00110101
00110110
00111001
00111010
00111100
01010011
01010101
01010110
01011010
01100011
01100101
01100110
01101001
01101010
01101100
01110001
01110010
01110100
01111000
10100011
10100101
10100110
10101010
10110001
10110010
10110100
10111000
11110000
Unique structures found 6 from 70
Necessary structures found 38 from 70
